# 2. Bilgi testi (TurkishMMLU)

[TurkishMMLU](https://huggingface.co/datasets/AYueksel/TurkishMMLU) Türk lise
müfredatından gerçek sınav sorularıdır (çeviri değil). 9 ders, 5 şık (A-E).

**Yöntem:** 5-shot, harf olasılığıyla puanlama. Model metin üretmez; `Cevap:`
ifadesinden sonraki ilk token için A-E harflerinin olasılıkları karşılaştırılır.
Deterministiktir ve cevap ayrıştırma hatası içermez.

Ders başına `ortak.N_PER_SUBJECT` soru sabit tohumla örneklenir; her model x seviye
koşusu birebir aynı soruları görür.

Veri seti erişim izni isterse Hugging Face sayfasından izin alıp `HF_TOKEN` ekleyin.

In [ ]:
!pip install -q -U "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43" "datasets>=2.20"

In [ ]:
import os
import sys
import time

import pandas as pd

# Depo kökünü bul: Colab'da /content/proje ya da /content, Kaggle'da /kaggle/...
for _kok in ["/content/proje", "/content", "/kaggle/working/proje", "/kaggle/working", "."]:
    if os.path.isdir(os.path.join(_kok, "kod")):
        sys.path.insert(0, os.path.join(_kok, "kod"))
        break
else:
    raise RuntimeError(
        "kod/ klasörü bulunamadı. Depoyu klonlayın:\n"
        "  !git clone https://github.com/<kullanici>/<depo>.git /content/proje"
    )

import ortak as O

O.set_seed()
O.hf_login()
print(O.env_report())
print("Çıktı klasörü:", O.OUT_DIR)

In [ ]:
MODEL_KEY = "trendyol"
RUN_VARIANTS = ["int8", "nf4"]

import turkishmmlu as T

print("Dersler:", T.load_subjects())

## Veri setini yükle ve ayrıştırmayı doğrula

Aşağıdaki çıktıda örnek satırın doğru ayrıştırıldığını kontrol edin: soru metni,
5 şık ve doğru cevabın indeksi tutarlı olmalı. Veri setinin sütun adları sürümler
arasında değişebildiği için `turkishmmlu.py` bunları otomatik buluyor.

In [ ]:
data = T.load_all()
items = T.eval_items(data)
print(f"\nDeğerlendirme sorusu: {len(items)}")
print("\n--- Örnek istem ---\n" + items[0]["prompt"])

In [ ]:
tok = O.load_tokenizer(MODEL_KEY)
letter_ids = O.letter_token_ids(tok, O.MMLU_LETTERS)
print("Harf token'ları:", dict(zip(O.MMLU_LETTERS, letter_ids)))

In [ ]:
for variant in RUN_VARIANTS:
    run = O.Run(MODEL_KEY, variant, "bilgi")
    if run.done():
        print(f"{run.name}: zaten var, atlanıyor")
        continue

    print(f"\n===== {MODEL_KEY} / {O.VARIANT_LABELS[variant]} =====")
    O.free_gpu()
    model = O.load_model(MODEL_KEY, variant)

    rows, t0 = [], time.perf_counter()
    for k, ev in enumerate(items, 1):
        pred, mass = O.score_letters(tok, model, ev["prompt"], letter_ids)
        it = ev["item"]
        rows.append({"model": MODEL_KEY, "variant": variant, "subject": ev["subject"],
                     "row": it["row"], "answer": it["answer"], "pred": pred,
                     "correct": pred == it["answer"], "letter_mass": round(mass, 4),
                     "difficulty": it["difficulty"]})
        if k % 100 == 0:
            acc = sum(r["correct"] for r in rows) / k
            print(f"  {k}/{len(items)}  doğruluk={acc:.3f}  {time.perf_counter() - t0:.0f} sn")

    df = pd.DataFrame(rows)
    df.to_csv(run.path(), index=False)
    print(f"Toplam doğruluk: {df.correct.mean():.3f}  ->  {run.path()}")

    del model
    O.free_gpu()

In [ ]:
# Bu koşunun ders bazında doğruluğu (şans seviyesi 5 şıkta %20)
if 'df' in dir():
    display((df.groupby("subject")["correct"].mean() * 100).round(1).to_frame("doğruluk_%"))